Load libraries and SciSpaCy model.

In [26]:
import os, re, json
import numpy as np
import spacy
import scispacy
from pathlib import Path
from tqdm import tqdm

Set data paths and load the 178 test PMIDs from CSV.

In [27]:
nlp = spacy.load("en_core_sci_sm")
print("SciSpaCy model loaded.")

c:\Users\harsh\OneDrive - University of Bristol\Desktop\EBM-NLP-master\venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


SciSpaCy model loaded.


Read each test abstract from disk into a dictionary.

In [ ]:
import pandas as pd

DATA_DIR    = Path("../../data/ebm_nlp_2_00")
ABSTRACTS_DIR = DATA_DIR / "documents"
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)


TEST_CSV  = RESULTS_DIR / "sentence_test_records_df.csv"
test_df   = pd.read_csv(TEST_CSV)
test_pmids = set(test_df["doc_id"].astype(str).unique())
print(f"Test PMIDs loaded: {len(test_pmids)}")


abstracts = {}
for pmid in test_pmids:
    filepath = ABSTRACTS_DIR / f"{pmid}.txt"
    if filepath.exists():
        with open(filepath, "r", encoding="utf-8") as f:
            abstracts[pmid] = f.read().strip()

print(f"Loaded {len(abstracts)} abstracts")

Test PMIDs loaded: 178
Loaded 178 abstracts


Define all regex patterns for PICO field extraction.

In [29]:
PATTERNS = {
    "participants": [
        r"(\d[\d,]*)\s*(patients|participants|subjects|adults|children|women|men|individuals|volunteers)",
        r"(patients|participants|subjects)\s*(with|who had|diagnosed with)\s+([A-Za-z\s]+)",
        r"(aged?\s+\d+[\s\-–to]+\d+\s*years?)",
        r"(men|women|adults|elderly|pediatric|geriatric)\s+(with|aged?)",
        r"(randomized|enrolled|recruited|assigned)\s+(\d[\d,]*)\s*(patients|participants|subjects)",
        r"(inclusion criteria|eligible patients|study population)[\:\s]+(.+?)(?=\.)",
        r"(patients with|adults with|children with|subjects with|individuals with)\s+([A-Za-z\s\-]+?)(?=\s*(?:,|\.|;|who|were|aged))",
        r"(diagnosed with|suffering from|presenting with)\s+([A-Za-z\s\-]+?)(?=\s*(?:,|\.|;|who|were))",
        r"(consecutive patients|eligible patients|study participants|trial participants)",
        r"(male|female)\s+(patients|participants|subjects|adults|children)",
        r"(mean age|average age)\s+(?:of|was|were)?\s*(\d+[\.\d]*)\s*(?:years|yr)",
    ],
    "intervention": [
        r"(treated with|received|administered|given|assigned to)\s+([A-Za-z0-9\-\s]+?)(?=\s*(?:vs|versus|or|compared|,|\.))",
        r"(intervention|treatment|therapy|drug|medication|dose|dosage)[\:\s]+([A-Za-z0-9\-\s]+?)(?=\.|\,)",
        r"([A-Z][a-z]+(?:mab|nib|zumab|tinib|ciclib|pril|sartan|olol|statin))",
        r"(surgery|surgical|operation|procedure|resection|transplant)",
        r"(mg|mcg|µg|IU)\s*(per|/|daily|twice|once)",
    ],
    "comparator": [
        r"(compared to|versus|vs\.?|compared with|in comparison to)\s+([A-Za-z0-9\-\s]+?)(?=\s*(?:,|\.|;|\())",
        r"(placebo|control group|sham|standard care|usual care|no treatment)",
        r"(control arm|control condition|reference group)",
        r"(randomized to|allocated to)\s+(?:either\s+)?([A-Za-z\s]+)\s+or\s+([A-Za-z\s]+)",
    ],
    "outcome": [
        r"(primary outcome|primary endpoint|primary efficacy)[\s\:was]+([^\.]+)",
        r"(secondary outcome|secondary endpoint)[\s\:was]+([^\.]+)",
        r"(we measured|we assessed|we evaluated|measured by|assessed by)\s+([^\.]+)",
        r"(was measured|were measured|was assessed|were assessed)\s+(?:by|using|with)\s+([^\.]+)",
        r"(significantly\s+(?:reduced|improved|increased|decreased|lower|higher|better|worse))\s+([^\.]+)",
        r"(no significant|significant\s+(?:difference|reduction|improvement|increase|decrease))\s*(?:in|between)?\s*([^\.]+)",
        r"(resulted in|leading to|associated with)\s+(?:a\s+)?(?:significant\s+)?([^\.]+?)(?=\s*(?:,|\.|;|\())",
        r"(mortality|survival|response rate|remission|recurrence|relapse|hospitalization|readmission)",
        r"(HbA1c|blood pressure|BMI|pain score|quality of life|QoL|adverse event|side effect)",
        r"(visual acuity|lung function|FEV1|CD4|viral load|tumour|tumor|lesion)",
        r"(statistically significant|p\s*[<>=]\s*0\.\d+)",
        r"(odds ratio|relative risk|hazard ratio|confidence interval|OR\s*=|RR\s*=|HR\s*=)\s*[\d\.\(\)\-\s]+",
        r"(mean\s+(?:difference|change|reduction|improvement)\s+(?:of|in|was))\s+([^\.]+)",
        r"(reduction in|improvement in|increase in|decrease in)\s+([A-Za-z\s]+?)(?=\s*(?:,|\.|;))",
        r"(reduced|improved|increased|decreased)\s+([A-Za-z\s]+?)\s+(?:by|from|compared)",
    ]
}

Function to run regex patterns and return deduplicated PICO matches.

In [30]:
def extract_pico_rule_based(pmid, text, nlp_model, patterns):
    doc = nlp_model(text)
    result = {
        "pmid": pmid,
        "participants": [],
        "intervention": [],
        "comparator": [],
        "outcome": []
    }

    for field, field_patterns in patterns.items():
        matches = []
        for pattern in field_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                span = match.group(0).strip()
                if span and len(span) > 3:  
                    matches.append(span)
        seen = set()
        deduped = []
        for m in matches:
            if m.lower() not in seen:
                seen.add(m.lower())
                deduped.append(m)
        result[field] = deduped

    return result

Function to add SciSpaCy named entities on top of regex results.

In [31]:
def enhance_with_scispacy(result, doc):
    for ent in doc.ents:
        if ent.label_ == "CHEMICAL":
            if ent.text.strip() not in result["intervention"]:
                result["intervention"].append(ent.text.strip())
        elif ent.label_ == "DISEASE":
            if ent.text.strip() not in result["outcome"]:
                result["outcome"].append(ent.text.strip())
    return result

Run extraction on all abstracts and save to JSON.

In [32]:
RESULTS_DIR = Path(r"C:\Users\harsh\OneDrive - University of Bristol\Desktop\EBM-NLP-master\ebm-nlp-project\results")
RESULTS_DIR.mkdir(exist_ok=True)

all_results = []

for pmid, text in tqdm(abstracts.items(), desc="Extracting PICO"):
    doc = nlp(text)
    result = extract_pico_rule_based(pmid, text, nlp, PATTERNS)
    result = enhance_with_scispacy(result, doc)
    all_results.append(result)

output_path = RESULTS_DIR / "rule_based_predictions.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved {len(all_results)} results to {output_path}")

Extracting PICO: 100%|██████████| 178/178 [00:47<00:00,  3.76it/s]

Saved 178 results to C:\Users\harsh\OneDrive - University of Bristol\Desktop\EBM-NLP-master\ebm-nlp-project\results\rule_based_predictions.json


Debug check to confirm gold files and abstracts align.

In [33]:
gold_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / "participants" / "test" / "gold"
gold_files = set(f.stem.replace(".AGGREGATED", "") for f in gold_path.glob("*.AGGREGATED.ann"))
test_pmids_in_abstracts = set(abstracts.keys())

print("Gold files count:", len(gold_files))
print("Test abstracts count:", len(test_pmids_in_abstracts))
print("Matching (in both):", len(gold_files & test_pmids_in_abstracts))
print("In gold but NOT in your abstracts:", len(gold_files - test_pmids_in_abstracts))
print("In your abstracts but NOT in gold:", len(test_pmids_in_abstracts - gold_files))

print("\nSample gold IDs:", list(gold_files)[:5])
print("Sample abstract IDs:", list(test_pmids_in_abstracts)[:5])

Gold files count: 189
Test abstracts count: 178
Matching (in both): 178
In gold but NOT in your abstracts: 11
In your abstracts but NOT in gold: 0

Sample gold IDs: ['11750293', '8913901', '26278470', '12139812', '15358868']
Sample abstract IDs: ['11750293', '8913901', '26278470', '12139812', '8989009']


Function to load gold token labels from annotation files.

In [34]:
def load_gold_labels(pmid, data_dir):
    labels_by_field = {
        "participants": [],
        "intervention": [],
        "outcome":      []
    }

    field_to_folder = {
        "participants": "participants",
        "intervention": "interventions",
        "outcome":      "outcomes",
    }


    token_path = data_dir / "documents" / f"{pmid}.tokens"
    if not token_path.exists():
       
        token_path = None

    for field, folder in field_to_folder.items():
        ann_path = (
            data_dir / "annotations" / "aggregated"
            / "hierarchical_labels" / folder
            / "test" / "gold" / f"{pmid}.AGGREGATED.ann"
        )
        if not ann_path.exists():
            continue

        with open(ann_path, "r", encoding="utf-8") as f:
            label_lines = [l.strip() for l in f.readlines()]

        if token_path and token_path.exists():
            with open(token_path, "r", encoding="utf-8") as f:
                tokens = [l.strip() for l in f.readlines()]
        else:
            tokens = ["tok"] * len(label_lines)  # placeholder if no token file

        for token, label in zip(tokens, label_lines):
            if label != "0":
                labels_by_field[field].append(token)

    return labels_by_field

Define stop words and token-level F1 scoring function.


Run evaluation loop and print P/R/F1 per field plus coverage.


Save predictions to CSV with standard column names.

In [35]:
import numpy as np
from tqdm import tqdm
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOP_WORDS = ENGLISH_STOP_WORDS

LABEL_TO_FIELD = {
    "1": "participants",
    "2": "intervention",
    "3": "outcome",
    "7": "comparator"
}

def load_gold_labels(pmid, data_dir):
    labels_by_field = {
        "participants": [],
        "intervention": [],
        "outcome":      []
    }

    field_to_folder = {
        "participants": "participants",
        "intervention": "interventions",
        "outcome":      "outcomes",
    }

    for field, folder in field_to_folder.items():
        ann_path = (
            data_dir / "annotations" / "aggregated"
            / "hierarchical_labels" / folder
            / "test" / "gold" / f"{pmid}.AGGREGATED.ann"
        )
        if not ann_path.exists():
            continue

        token_path = data_dir / "documents" / f"{pmid}.tokens"

        with open(ann_path, "r", encoding="utf-8") as f:
            label_lines = [l.strip() for l in f.readlines()]

        with open(token_path, "r", encoding="utf-8") as f:
            tokens = [l.strip() for l in f.readlines()]

        for token, label in zip(tokens, label_lines):
            if label != "0":
                labels_by_field[field].append(token)

    return labels_by_field


def token_f1(pred_spans, gold_tokens):
    pred_tokens = set(" ".join(pred_spans).lower().split()) if pred_spans else set()
    gold_set    = set(t.lower() for t in gold_tokens) if gold_tokens else set()
    pred_tokens = pred_tokens - STOP_WORDS
    gold_set    = gold_set - STOP_WORDS
    if not gold_set:
        return None, None, None
    if not pred_tokens:
        return 0.0, 0.0, 0.0
    tp        = len(pred_tokens & gold_set)
    precision = tp / len(pred_tokens)
    recall    = tp / len(gold_set)
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

fields  = ["participants", "intervention",  "outcome"]
TOTAL_TEST_DOCS = 178
scores  = {f: {"precision": [], "recall": [], "f1": []} for f in fields}
coverage_count  = 0
evaluated_count = 0

for result in tqdm(all_results, desc="Evaluating"):
    pmid = result["pmid"]
    gold = load_gold_labels(pmid, DATA_DIR)
    all_filled = True

    for field in fields:
        p, r, f = token_f1(result[field], gold.get(field, []))

        if p is None:
            continue

        scores[field]["precision"].append(p)
        scores[field]["recall"].append(r)
        scores[field]["f1"].append(f)

        if not result[field]:
            all_filled = False

    if all_filled:
        coverage_count += 1
    evaluated_count += 1


print("\n=== Rule-Based Pipeline Evaluation ===")
for field in fields:
    if not scores[field]["f1"]:
        print(f"{field.upper():<15} No gold data found")
        continue
    avg_p = np.mean(scores[field]["precision"])
    avg_r = np.mean(scores[field]["recall"])
    avg_f = np.mean(scores[field]["f1"])
    print(f"{field.upper():<15} P={avg_p:.3f}  R={avg_r:.3f}  F1={avg_f:.3f}")

valid_fields = [f for f in fields if scores[f]["f1"]]
all_f1s      = [np.mean(scores[f]["f1"]) for f in valid_fields]
print(f"\nAverage F1:  {np.mean(all_f1s):.3f}")
print(f"Coverage:    {coverage_count}/{TOTAL_TEST_DOCS} = {coverage_count/TOTAL_TEST_DOCS*100:.1f}%")


rows = []
for r in all_results:
    rows.append({
        "doc_id":              r["pmid"],
        "participants_pred":  " ; ".join(r["participants"]) if r["participants"] else None,
        "interventions_pred":  " ; ".join(r["intervention"]) if r["intervention"] else None,
        "outcomes_pred":  " ; ".join(r["outcome"]) if r["outcome"]      else None
})

df_out   = pd.DataFrame(rows)
csv_path = RESULTS_DIR / "axis1A_rule_based_pred.csv"
df_out.to_csv(csv_path, index=False)
print(f"\nCSV saved → {csv_path}")
print(df_out.head(3))

Evaluating: 100%|██████████| 178/178 [00:00<00:00, 213.16it/s]


=== Rule-Based Pipeline Evaluation ===
PARTICIPANTS    P=0.486  R=0.316  F1=0.337
INTERVENTION    P=0.112  R=0.189  F1=0.115
OUTCOME         P=0.205  R=0.202  F1=0.172

Average F1:  0.208
Coverage:    89/178 = 50.0%

CSV saved → C:\Users\harsh\OneDrive - University of Bristol\Desktop\EBM-NLP-master\ebm-nlp-project\results\axis1A_rule_based_pred.csv
     doc_id                                  participants_pred  \
0  11750293  22 patients ; patients with recently diagnosed...   
1   8913901  568 participants ; patients with multivessel c...   
2  26278470  90 patients ; patients with OCD with poor insi...   

                                  interventions_pred  \
0  received standard outpatient clinic treatment ...   
1                                  received at the C   
2  received either 24 CBT sessions ; treatment sp...   

                                       outcomes_pred  
0                                                NaN  
1                                               